# 📗 기준선과 pytest·CI로 RAG 회귀 막기

도서관 안내 RAG 챗봇에 "응답 비용을 줄이려고 검색 문서 수를 3에서 1로 줄이자"는 변경 요청(pull request)이 들어왔습니다. 이 변경을 받아들여도 될까요?

질문 몇 개를 넣어 보는 것으로는 답할 수 없습니다. 이 교안은 지금 설정의 평가 결과를 **기준선**으로 정하고, 새 설정이 기준선보다 나빠지면(회귀) 실패하는 pytest 테스트를 만들고, 변경 요청이 올라올 때마다 CI(지속적 통합)가 이 테스트를 자동으로 돌리게 합니다. 이렇게 품질이 떨어진 변경을 막는 장치를 **회귀 게이트**라고 합니다.

**학습 목표**

1. **같은 설정을 반복 평가해 판정이 달라지는 정도(흔들림)를 계산합니다.** `start_run` 안에서 `log_param`·`log_metric`으로 실행 조건과 평가 결과를 한 run에 남깁니다.
2. **승인한 run을 기준선으로 정하고 판정 규칙 네 가지를 세웁니다.** 허용 하락폭은 계산한 지표 흔들림보다 크게 정합니다.
3. **변경 설정을 평가해 기준선과 사례 단위로 비교합니다.** `search_runs`로 run을 표로 모으고 `case_id`로 판정을 맞춥니다.
4. **평가 결과를 판정하는 pytest 회귀 테스트를 작성하고 실행합니다.** 실패하면 종료 코드가 0이 아니게 되는 원리를 확인하고, DeepEval의 `assert_test` 방식과 비교합니다.
5. **GitHub Actions에서 변경 요청마다 평가와 테스트를 돌립니다.** API 키는 비밀 변수로 넘기고 결과는 요약과 산출물 파일로 남깁니다.

<img src="../images/06_ci_workflow.png" width="1050" alt="GitHub Actions가 변경 요청을 받아 라이브러리 설치, 평가 실행, 회귀 테스트, 요약·산출물 저장 순서로 실행하는 흐름입니다. 변경 요청은 대표 6건, 매일 예약 실행은 전체 16건을 평가하고 API 키는 비밀 변수로 넘깁니다.">

## 1. 실습 준비

### 1-1. 라이브러리·모델·MLflow 설정

교안 01과 같은 설정입니다. 결과도 같은 `mlflow.db` 파일의 `library-rag-eval` 실험에 기록합니다.

In [ ]:
# 실습에 필요한 라이브러리를 불러옵니다.
import os
import warnings
from pathlib import Path

import mlflow
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# DeepEval 이 사용 기록을 외부로 보내지 않게 합니다. deepeval 을 불러오기 전에 정해야 합니다.
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"

# 채점 중 나오는 ipywidgets 설치 권고 경고를 숨깁니다.
warnings.filterwarnings("ignore", message='install "ipywidgets"')
from deepeval.models import OpenAIModel

# material_dir 는 data 와 mlflow.db 가 있는 단원 폴더입니다.
material_dir = Path("..")
data_dir = material_dir / "data"

load_dotenv(find_dotenv(usecwd=True))
# 단원 .env에 없는 공통 키는 저장소 최상위 .env에서 읽습니다. 이미 읽은 키는 덮어쓰지 않습니다.
load_dotenv(material_dir.resolve().parent / ".env")

# 답변 모델: 챗봇 답변을 만듭니다.
model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-4.1")
llm = ChatOpenAI(model=model_name, use_responses_api=True)

# 미리 만들어 둔 문서 벡터와 같은 모델·차원으로 질문만 임베딩합니다.
embedding_model = OpenAIEmbeddings(model="text-embedding-3-large", dimensions=768,
                                   check_embedding_ctx_length=False)  # 짧은 질문이라 자동 길이 검사·분할을 끕니다.

# 판사 모델: 답변 모델과 다른 모델로 채점합니다. 판사를 바꾸면 기준선도 다시 만들어야 합니다.
# gpt-5.6-luna 는 temperature 를 지정할 수 없는 모델이라 지정하지 않습니다.
eval_model_name = os.getenv("OPENAI_EVAL_MODEL", "gpt-5.6-luna")
judge_llm = OpenAIModel(model=eval_model_name)

# MLflow 자동 기록(autolog)을 끕니다. 켜져 있으면 호출마다 실행 기록(trace)에 단계가 자동으로 생깁니다.
# - mlflow.langchain.autolog: LangChain 체인·프롬프트·모델 호출을 단계마다 입력, 출력, 걸린 시간, 토큰 수와 함께 기록합니다.
# - mlflow.openai.autolog: OpenAI SDK 호출(DeepEval 판사의 호출 포함)을 보낸 메시지, 모델 이름, 응답, 토큰 수와 함께 기록합니다.
# 꺼 두지 않으면 mlflow.genai.evaluate 가 평가 중에 이 둘을 잠시 켭니다. MLflow 3.14 에서는 그 기록이 늦게 닫히면 평가가 실행 기록을 찾지 못해 오류로 멈출 때가 있습니다.
# 실행 기록은 @mlflow.trace 로 직접 남깁니다.
mlflow.langchain.autolog(disable=True)
mlflow.openai.autolog(disable=True)

# 평가 run 과 실행 기록은 단원 폴더의 mlflow.db 파일에 쌓습니다.
mlflow.set_tracking_uri(f"sqlite:///{material_dir / 'mlflow.db'}")
experiment = mlflow.set_experiment("library-rag-eval")

print("답변 모델:", model_name, "/ 판사 모델:", eval_model_name, "/ 실험:", experiment.name)

### 1-2. 교안 01의 평가 부품

아래 여섯 셀은 교안 01과 같은 코드라 차례로 실행만 하면 됩니다. 채점 함수 셀 끝에만 판정에 쓸 목록(`gate_scorers`, `JUDGED_SCORERS`)을 더했습니다.

이 교안에서 다시 쓰는 이름은 다음과 같습니다.

| 이름 | 하는 일 |
|---|---|
| `library_records` | 골든셋(정답셋)을 `evaluate`가 읽는 입력 형식으로 바꾼 목록 |
| `answer_with_citations(question, top_k=3)` | 문서를 검색해 인용을 붙여 답하고 `segments`, `answer`, `searched_ids`를 담은 dict를 돌려줌 |
| `gate_scorers`, `JUDGED_SCORERS` | 판정에 쓰는 채점 함수 목록(`correctness` 정답 판정, `citation_valid` 인용 검증)과 그 이름 |

`CORRECTNESS_STEPS`와 `judge_llm`은 DeepEval `assert_test` 비교에서 씁니다. 표에 없는 이름(`read_jsonl`, `to_records`, `check_citation` 등)은 7절 HR 상담 챗봇 부품 셀에서 씁니다.

**문서와 검색**: 이용 안내 문서를 읽고, `search_documents`가 질문과 가까운 문서를 찾습니다.

In [ ]:
import json

from langchain_chroma import Chroma

# read_jsonl: data 폴더의 jsonl 파일(한 줄에 JSON 하나)을 목록으로 읽습니다.
def read_jsonl(name):
    rows = []
    for line in (data_dir / name).read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows


library_docs = read_jsonl("본실습/docs.jsonl")

# 인용을 검증할 때 근거 ID로 원문을 찾을 수 있게 사전으로 둡니다.
library_texts = {}
for doc in library_docs:
    library_texts[doc["doc_id"]] = doc["text"]

vector_store = Chroma(collection_name="library_docs", embedding_function=embedding_model,
                      persist_directory=str(data_dir / "본실습" / "chroma"))

print("이용 안내 문서:", len(library_docs))

@mlflow.trace(span_type="RETRIEVER")
def search_documents(search_query, top_k=3):
    """질문과 가까운 안내 문서 top_k 개를 Document 목록으로 반환합니다."""
    return vector_store.similarity_search(search_query, k=top_k)


def format_documents(docs):
    """검색 문서를 [근거 ID] 제목과 본문 형식으로 이어 붙입니다."""
    parts = []
    for doc in docs:
        parts.append(f"[{doc.metadata['source_id']}] {doc.metadata['title']}\n{doc.page_content}")
    return "\n\n".join(parts)

**골든셋 입력**: 골든 사례를 읽어 `evaluate`의 입력 형식(`library_records`)으로 바꿉니다.

In [ ]:
library_golden = read_jsonl("본실습/golden.jsonl")


def to_records(cases):
    """골든 사례를 evaluate 가 읽는 inputs·expectations 형식으로 바꿉니다."""
    records = []
    for case in cases:
        # inputs 의 키는 답변 함수의 인자 이름(question)과 같아야 합니다.
        inputs = {"question": case["input"]}
        expectations = {"case_id": case["case_id"], "expected_output": case["expected_output"]}
        records.append({"inputs": inputs, "expectations": expectations})
    return records


library_records = to_records(library_golden)

print("골든 사례:", len(library_records))

**인용 답변 함수**: 검색한 문서로 답하고 답변 부분마다 근거 ID와 원문 구절을 붙입니다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field

class Citation(BaseModel):
    evidence_id: str = Field(description="근거 ID를 넣습니다. 제공된 ID를 철자 그대로 씁니다")
    quote: str = Field(description="그 근거 원문에서 글자 그대로 복사한 구절을 넣습니다. 고치거나 요약하지 않습니다")


class Segment(BaseModel):
    text: str = Field(description="답변의 한 부분(보통 문장 하나)을 넣습니다. segments 의 text 를 순서대로 이으면 사용자에게 보여 줄 답변이 됩니다")
    citations: list[Citation] = Field(description="이 부분을 뒷받침하는 인용 목록입니다. 문서에서 가져온 사실이 아니면(연결 문장, 확인할 수 없다는 안내 등) 빈 목록입니다")


class CitedAnswer(BaseModel):
    segments: list[Segment] = Field(description="답변을 순서대로 나눈 부분 목록입니다")


cited_prompt = ChatPromptTemplate.from_messages([
    ("system", "도서관 이용 안내 문서로만 답하세요. 답변을 부분(segments)으로 나눠 쓰고, 문서에서 가져온 사실을 쓴 부분마다 근거 ID와 "
               "그 문서에서 글자 그대로 복사한 구절을 인용하세요. 구절을 바꿔 쓰거나 요약하지 마세요. 조건과 예외가 있으면 함께 쓰세요. "
               "문서에 답이 없으면 '제공된 이용 안내에서 확인할 수 없습니다.'라는 부분 하나만 인용 없이 쓰세요."),
    ("human", "질문: {question}\n\n문서:\n{context}"),
])

cited_chain = cited_prompt | llm.with_structured_output(CitedAnswer)


@mlflow.trace
def answer_with_citations(question, top_k=3):
    """질문으로 문서를 검색하고, 답변 부분마다 근거 ID 와 원문 구절을 인용으로 붙여 답합니다.

    question: 사용자 질문입니다. evaluate 의 inputs["question"] 이 이 인자로 들어옵니다.
    top_k: 검색해 올 문서 수입니다.
    반환값: dict 입니다. 채점 함수는 이 dict 를 outputs 로 받습니다.
        segments: 답변 부분 목록. 부분마다 text(문장)와 citations(근거 ID·인용 구절 목록)가 있습니다.
        answer: 부분의 text 를 이어 붙인, 사용자에게 보여 줄 답변입니다.
        searched_ids: 이번에 검색해 온 문서의 근거 ID 목록입니다.
    """
    docs = search_documents(question, top_k)
    result = cited_chain.invoke({"question": question, "context": format_documents(docs)})

    # 구조화 출력 객체를 dict 로 바꿉니다. 채점 함수는 이 dict 를 outputs 로 읽습니다.
    output = result.model_dump()

    # 부분을 순서대로 이어 사용자에게 보여 줄 답변을 만듭니다.
    texts = []
    for segment in output["segments"]:
        texts.append(segment["text"])
    output["answer"] = " ".join(texts)

    # 검색한 근거 ID도 넣어야 "검색하지 않은 문서를 인용했는지" 검사할 수 있습니다.
    output["searched_ids"] = []
    for doc in docs:
        output["searched_ids"].append(doc.metadata["source_id"])
    return output

**인용 검사**: 인용 하나마다 근거 ID와 인용 문구를 차례로 확인합니다.

In [ ]:
def normalize(text):
    """뜻과 상관없는 표기 차이(가운뎃점 모양, 줄바꿈, 연속 공백)를 하나로 맞춥니다."""
    for mark in ["･", "∙", "・", "•"]:
        text = text.replace(mark, "·")

    # split 은 줄바꿈·탭·연속 공백에서 모두 나누므로, 공백 하나로 다시 이으면 공백이 통일됩니다.
    return " ".join(text.split())


def check_citation(citation, source_texts, searched_ids):
    """인용 하나가 이번 답변의 실제 근거인지 차례로 검사해, 처음 걸린 문제 또는 "확인"을 돌려줍니다.

    citation: 답변 부분에 붙은 인용 하나입니다. {"evidence_id": 근거 ID, "quote": 인용 구절}
    source_texts: 근거 ID -> 문서 원문 사전입니다.
    searched_ids: 이번 질문에서 검색해 온 문서의 근거 ID 목록입니다.
    반환값: 아래 순서로 검사해 처음 걸린 문제를 돌려줍니다. 모두 통과하면 "확인"입니다.
        "없는 근거 ID": 문서에 없는 근거 ID 를 인용함
        "검색하지 않은 문서": 있는 문서지만 이번 질문에서 검색하지 않음
        "원문에 없는 문구": 표기를 맞춘(normalize) 뒤에도 인용 구절이 원문에 없음
    """
    cited_id = citation["evidence_id"]
    if cited_id not in source_texts:
        return "없는 근거 ID"
    if cited_id not in searched_ids:
        return "검색하지 않은 문서"
    if normalize(citation["quote"]) not in normalize(source_texts[cited_id]):
        return "원문에 없는 문구"
    return "확인"

**채점 함수**: 판정에는 두 채점 함수만 씁니다. 교안 01의 검색 근거 판정(`faithfulness`)은 판사를 여러 번 불러 비용이 커서 판정 목록에서 뺐습니다.

두 채점 함수는 교안 01과 같습니다.

In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, SingleTurnParams
from mlflow.entities import Feedback
from mlflow.genai.scorers import scorer

# 정답 판정의 판단 단계입니다. 정답 답변(expected_output)과 비교합니다.
CORRECTNESS_STEPS = [
    "expected_output 에 적힌 사실이 actual_output 에 모두 들어 있는지 본다.",
    "숫자, 기간, 대상, 조건이 하나라도 다르거나 빠지면 틀린 것으로 본다.",
    "expected_output 이 자료에서 확인할 수 없다는 안내이면, actual_output 도 확인할 수 없다고 답해야 맞다.",
    "판정 이유(reason)는 한국어로 쓴다.",
]


@scorer
def correctness(inputs, outputs, expectations):
    """정답 판정: 답변이 정답 답변(expected_output)의 사실을 모두 담았는지 G-Eval 판사로 판정해, True/False 와 이유를 Feedback 으로 돌려줍니다."""
    # 정답 답변입니다. 자료에 없는 질문이면 확인할 수 없다는 안내 문장입니다.
    expected = expectations["expected_output"]

    # 평가는 여러 질문을 동시에 채점하므로, 채점 함수가 불릴 때마다 GEval 지표를 새로 만들어 점수가 섞이지 않게 합니다.
    metric = GEval(
        name="정답 판정",
        evaluation_steps=CORRECTNESS_STEPS,
        evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT],
        model=judge_llm,
        threshold=0.7,
        async_mode=False,  # 지표 안에서 비동기로 호출하지 않습니다(동시 채점 중 이벤트 루프 오류를 막습니다).
    )

    # 여러 질문을 동시에 채점하므로 진행 표시를 끕니다(켜 두면 표시가 서로 겹칩니다).
    metric.measure(LLMTestCase(input=inputs["question"], actual_output=outputs["answer"], expected_output=expected), _show_indicator=False)
    return Feedback(value=metric.success, rationale=metric.reason)

@scorer
def citation_valid(outputs):
    """인용 검증: 답변에 붙은 인용이 모두 check_citation 을 통과하면 True, 아니면 False 와 걸린 인용 목록을 Feedback 으로 돌려줍니다."""
    problems = []
    for segment in outputs["segments"]:
        for citation in segment["citations"]:
            status = check_citation(citation, library_texts, outputs["searched_ids"])
            if status != "확인":
                problems.append(f"{status}: {citation['evidence_id']}")

    if len(problems) == 0:
        return Feedback(value=True, rationale="붙은 인용이 모두 근거 ID·문구 대조를 통과함")
    return Feedback(value=False, rationale="; ".join(problems))

# 회귀 게이트가 통과·실패를 판정할 때 쓰는 채점 함수(gate_scorers)와 그 이름 목록(JUDGED_SCORERS)입니다.
# 보고서·기준선·테스트가 이 이름으로 지표를 찾으므로 @scorer 함수 이름과 같아야 합니다.
gate_scorers = [correctness, citation_valid]
JUDGED_SCORERS = ["correctness", "citation_valid"]

print("판정 채점 함수:", JUDGED_SCORERS)

## 2. 현재 설정을 세 번 평가하기

변경을 평가하기 전에 지금 서비스 중인 설정(현재 설정, `top_k=3`)이 어떤 결과를 내는지부터 알아야 합니다. 같은 설정이어도 판정이 run마다 달라질 수 있기 때문입니다(이유는 2-3).

그래서 같은 설정을 세 번 평가해 판정이 달라지는 정도(**흔들림**)를 먼저 계산합니다.

### 2-1. run 안에 조건과 평가를 함께 기록

`with mlflow.start_run(run_name=...)`으로 run을 열고 실행 조건을 남긴 뒤 그 안에서 평가합니다. 열린 run이 있으면 `evaluate`는 새 run을 만들지 않고 그 run에 결과를 씁니다. 실행 조건과 평가 결과가 같은 run에 들어가므로, 뒤에서 run을 조건별로 골라 비교할 수 있습니다.

In [ ]:
def evaluate_run(run_name, dataset, predict_fn, scorers, params):
    """run 을 하나 열어 실행 조건(params)을 기록하고, 같은 run 안에서 평가해 평가 결과를 돌려줍니다.

    run_name: 화면에 보일 run 이름입니다.
    dataset, predict_fn, scorers: mlflow.genai.evaluate 에 그대로 넘기는 평가 데이터, 답변 함수, 채점 함수 목록입니다.
    params: run 에 남길 실행 조건 dict 입니다(예: {"top_k": 3}).
    반환값: mlflow.genai.evaluate 의 평가 결과(metrics, result_df, run_id)입니다.
    """
    with mlflow.start_run(run_name=run_name):
        for key, value in params.items():
            mlflow.log_param(key, value)

        result = mlflow.genai.evaluate(data=dataset, predict_fn=predict_fn, scorers=scorers)
        mlflow.log_metric("case_count", len(result.result_df))
    return result

### 2-2. 같은 설정으로 세 run 평가

같은 설정을 `top3-a`, `top3-b`, `top3-c` 세 run으로 평가합니다(사례 16건씩). 첫 run `top3-a`가 뒤에서 승인할 기준선 run입니다.

In [ ]:
# 현재 설정(지금 서비스 중인 설정)입니다. 뒤에서 평가할 변경 요청은 이 가운데 top_k 만 바꿉니다.
top3_params = {"prompt_version": "v1", "app_model": model_name, "judge_model": eval_model_name, "top_k": 3}

top3_results = []
for run_name in ["top3-a", "top3-b", "top3-c"]:
    top3_results.append(evaluate_run(run_name, library_records, answer_with_citations, gate_scorers, top3_params))

print("평가한 run:", len(top3_results))

### 2-3. 사례별 흔들림 계산

설정도 질문도 같은데 판정이 run마다 달라질 수 있습니다. 평가 한 번에 LLM이 두 군데 들어가기 때문입니다.

| 흔들리는 곳 | 무엇이 달라지나 | 영향을 받는 채점 함수 |
|---|---|---|
| 답변 모델 | 같은 질문에도 문장, 인용 구절, 덧붙이는 내용이 매번 조금씩 다르게 나옵니다 | `correctness`, `faithfulness`, `citation_valid` |
| 판사 모델 | 같은 답변을 채점해도 판사의 출력이 매번 조금씩 다릅니다. G-Eval 점수나 Faithfulness의 주장 나누기·주장별 판정이 달라져, 점수가 합격선 0.7 근처면 통과와 실패가 바뀝니다 | `correctness`, `faithfulness` |

`citation_valid`는 코드라서 같은 답변에는 늘 같은 판정을 냅니다. 이 채점 함수의 판정이 바뀌었다면 답변이 달라진 것입니다.

이 교안은 비용 때문에 `faithfulness`를 빼고 두 채점 함수로만 판정합니다(1절).

흔들림을 모르면 멀쩡한 변경을 막게 됩니다. 원래 세 번 중 두 번만 통과하던 사례가 변경 뒤에 한 번 실패했다면, 그것은 변경 탓이 아니라 원래 있던 흔들림입니다. 그래서 같은 설정을 세 번 평가해 두 가지를 구하고 3절의 규칙에 씁니다.

- **늘 통과한 사례**: 두 채점 함수 모두에서 세 번 다 통과한 사례입니다. 이 사례가 실패하면 흔들림이 아니라 회귀로 봅니다(사례별 회귀 규칙, 3-1).
- **지표 흔들림**: 세 run 지표의 최댓값 - 최솟값입니다. 지표가 이만큼 떨어지는 것은 흔들림으로 보고 허용합니다(허용 하락폭, 3-1).

아래 코드는 `make_report`로 세 run의 결과를 사례별 통과 여부와 지표로 정리한 뒤, 사례마다 세 번 중 몇 번 통과했는지 셉니다.

<img src="../images/08_repeat_runs_stable_cases.png" width="1050" alt="같은 설정을 세 번 평가해 사례마다 통과 횟수를 셉니다. 3/3은 늘 통과한 사례로 사례별 회귀 규칙에 쓰고, 2/3은 흔들린 사례라 규칙에서 빼고, 0/3은 늘 실패한 사례라 개선 과제로 따로 둡니다. 아래 띠는 지표 흔들림과 허용 하락폭 식입니다.">

In [ ]:
import pandas as pd

def pass_rates(cases, case_ids):
    """case_ids 사례 가운데 채점 함수마다 통과한 사례의 비율을 계산합니다.

    cases: make_report 가 만든 사례별 판정입니다. {case_id: {채점 함수 이름: True/False, 오류면 None}}
    case_ids: cases 가운데 비율을 계산할 사례 ID 목록입니다. 전체를 셀 때는 list(cases), 빠른 점검(smoke) 사례만 셀 때는 그 ID 목록을 넘깁니다.
    반환값: {채점 함수 이름: 통과 비율(0~1, 소수 넷째 자리까지)}. True 인 사례만 통과로 셉니다.
    """
    rates = {}
    for name in JUDGED_SCORERS:
        passed = 0
        for case_id in case_ids:
            if cases[case_id][name] is True:
                passed += 1
        rates[name] = round(passed / len(case_ids), 4)
    return rates


def make_report(result, split):
    """평가 결과를 사례별 통과 여부와 지표로 정리한 보고서 dict 를 돌려줍니다. 이 보고서를 파일로 저장해 두면 뒤에서 테스트가 읽습니다.

    result: mlflow.genai.evaluate 가 돌려준 평가 결과입니다.
    split: 평가 범위 이름입니다("full" 은 전체 16건, "smoke" 는 대표 6건).
    반환값:
        split, run_id: 평가 범위와 평가 run 의 ID
        errors: 판정이 비어 있는(채점 중 오류가 난) 칸 수
        metrics: 채점 함수마다 통과 비율(pass_rates)
        cases: {case_id: {채점 함수 이름: True/False, 오류면 None}}
    """
    cases = {}
    errors = 0
    for _, row in result.result_df.iterrows():
        case = {}
        for name in JUDGED_SCORERS:
            value = row[f"{name}/value"]
            # 채점 중 오류가 난 행은 값이 비어 있습니다. 통과로 세지 않고 오류로 셉니다.
            if pd.isna(value):
                errors += 1
                case[name] = None
            else:
                # 채점 함수는 True/False 로 판정을 남깁니다. numpy 의 참거짓도 bool 로 맞춥니다.
                case[name] = bool(value == True)
        cases[row["case_id/value"]] = case
    return {"split": split, "run_id": result.run_id, "errors": errors,
            "metrics": pass_rates(cases, list(cases)), "cases": cases}

repeat_reports = []
for run_result in top3_results:
    repeat_reports.append(make_report(run_result, "full"))

# 첫 run(top3-a)의 보고서입니다. 세 보고서 모두 이 모양입니다.
display(repeat_reports[0])

In [ ]:
# 세 run 의 지표(채점 함수별 통과 비율)입니다.
for report in repeat_reports:
    print("지표:", report["metrics"])

`make_report`의 둘째 인자 `"full"`은 평가 범위(split)가 16건 전체라는 표시입니다. 대표 6건 범위(smoke)는 3-1에서 다룹니다.

In [ ]:
# 사례마다 세 번 중 몇 번 통과했는지 셉니다.
rows = []
for case in library_golden:
    case_id = case["case_id"]
    row = {"case_id": case_id}
    for name in JUDGED_SCORERS:
        passed = 0
        for report in repeat_reports:
            if report["cases"][case_id][name] is True:
                passed += 1
        row[name] = f"{passed}/3"
    rows.append(row)

display(pd.DataFrame(rows))

- **`1/3`이나 `2/3`인 칸**: 같은 설정에서 판정이 흔들린 사례입니다. 늘 통과한 사례가 아니므로 사례별 회귀 규칙에서 빠집니다.
- **`0/3`인 칸**: 현재 설정에서 늘 실패하는 사례입니다. 예를 들어 신간 코너 도서 연장 질문(`lib-13`)은 근거 문서 `loan-new-books`가 검색 4위라 `top_k=3` 안에 들지 않습니다.

늘 실패하는 사례는 회귀가 아니라 검색을 고쳐야 할 개선 과제이며, 사례별 회귀 규칙에서도 빠집니다. 모든 칸이 `3/3`이나 `0/3`이면 이번 세 번에서는 판정이 흔들린 사례가 없었던 것입니다. 다만 세 번만으로 흔들림이 없다고 단정할 수는 없습니다.

## 3. 기준선 정하기

흔들림을 알았으니, 이제 어느 run을 기준선으로 삼고 얼마나 떨어지면 막을지 정합니다.

**기준선**(baseline)은 팀이 검토해 서비스해도 되는 품질로 승인한 평가 결과입니다. 새 변경의 결과를 이 기준선과 비교해 통과 여부를 정합니다.

지표가 가장 높은 run을 자동으로 고르지 않고 사람이 결과를 읽고 승인합니다. 여기서는 첫 run `top3-a`를 기준선 run으로 씁니다.

### 3-1. 판정 규칙 네 가지

<img src="../images/05_baseline_gate_rules.png" width="1050" alt="절대 하한, 허용 하락폭, 사례별 회귀, 평가 완전성 네 규칙이 각각 어떤 경우를 실패로 판정하는지 보여 줍니다.">

한 가지 규칙만으로는 놓치는 회귀가 있어서 네 규칙을 함께 씁니다.

| 규칙 | 실패 조건 | 잡는 문제 |
|---|---|---|
| 절대 하한 | 지표가 팀이 정한 최소값보다 낮음 | 허용 하락폭 안이지만 최소 품질보다 낮아진 경우 |
| 허용 하락폭 | 지표가 기준선 값보다 허용 하락폭을 넘게 떨어짐 | 평균의 큰 하락 |
| 사례별 회귀 | 늘 통과한 사례가 실패 | 평균에는 드러나지 않는 개별 사례의 실패 |
| 평가 완전성 | 평가한 사례 목록이 기준선과 다르거나 채점 오류가 있음 | 일부 사례만 채점했는데 통과로 판정되는 경우 |

**허용 하락폭 = 지표 흔들림 + 1 / 사례 수**입니다. 사례 수는 그 평가 범위에서 채점한 사례 수(전체 16건, 대표 6건)입니다. 앞 항은 같은 설정 세 run 지표의 최댓값 - 최솟값으로 계산한 값이고, 뒤 항은 세 번만으로는 흔들림을 작게 볼 수 있어 사례 1건만큼 더 봐주는 여유입니다. 여유의 크기는 팀이 정합니다.

- **16건 전체(full)**: 1/16 = 0.0625(6.25%p)라서, 지표 흔들림이 0이어도 지표가 1건만큼 떨어진 것은 이 규칙을 통과합니다. 그 1건이 늘 통과한 사례라면 사례별 회귀 규칙에서 실패하므로, 1건 여유가 실제로 허용되는 것은 판정이 흔들리던 사례가 실패한 경우뿐입니다.
- **대표 6건(smoke)**: 변경 요청마다 빠르게 점검하려고 질문 유형(단일 사실, 여러 문서를 합쳐야 답하는 질문, 예외 조건, 자료에 없는 질문)마다 대표만 고른 묶음입니다. 1/6 = 0.1667이라 허용 하락폭을 따로 계산합니다.
- **절대 하한**: 서비스에 필요한 최소 품질이라 팀이 정하며, RAGAS 단원의 합격선과 같은 개념입니다. 현재 설정을 다시 평가해도 이 규칙에서 실패하지 않도록, 반복 평가에서 나온 가장 낮은 값보다 여유를 두고 정합니다.
- 이 교안은 정답 판정 0.6, 인용 검증 0.8로 둡니다. 현재 설정을 다시 평가할 때마다 정답 판정 지표가 0.7~0.9 사이에서 달라져, 가장 낮게 나온 값보다 여유를 두었습니다.

기준선은 세 단계로 만듭니다. 늘 통과한 사례를 찾고, 평가 범위마다 기준값을 계산한 뒤, 하나의 dict로 묶어 파일로 저장합니다.

**1단계 · 늘 통과한 사례**: 반복한 세 run 모두에서 두 채점 함수의 판정이 `True`인 사례입니다. 사례별 회귀 규칙에 씁니다.

In [ ]:
library_ids = []
for case in library_golden:
    library_ids.append(case["case_id"])

# 세 run 가운데 한 번이라도, 두 채점 함수 가운데 하나라도 True 가 아니면 늘 통과한 사례가 아닙니다.
library_stable_cases = []
for case_id in library_ids:
    passed_all = True
    for report in repeat_reports:
        for name in JUDGED_SCORERS:
            if report["cases"][case_id][name] is not True:
                passed_all = False
    if passed_all:
        library_stable_cases.append(case_id)

print("늘 통과한 사례:", len(library_stable_cases), "/", len(library_ids), library_stable_cases)

**2단계 · 평가 범위마다 기준값**: 16건 전체(`full`)와 대표 6건(`smoke`)마다 기준선 run(첫 run `top3-a`)의 통과 비율(`metrics`), 지표 흔들림(`noise`), 허용 하락폭(`allowed_drop`), 절대 하한(`absolute_floor`)을 계산합니다.

In [ ]:
library_smoke_ids = []
for case in library_golden:
    if case["smoke"]:
        library_smoke_ids.append(case["case_id"])

# 절대 하한은 팀이 정하는 최소 품질입니다.
library_absolute_floor = {"correctness": 0.6, "citation_valid": 0.8}

library_splits = {}
for split_name, case_ids in [("full", library_ids), ("smoke", library_smoke_ids)]:
    noise = {}
    allowed_drop = {}
    for name in JUDGED_SCORERS:
        # 세 run 의 통과 비율
        values = []
        for report in repeat_reports:
            values.append(pass_rates(report["cases"], case_ids)[name])

        # 흔들림 = 가장 높은 값 - 가장 낮은 값(계산한 값)
        # 허용 하락폭 = 흔들림 + 1 / 사례 수(사례 1건만큼의 여유, 팀이 정하는 값)
        noise[name] = round(max(values) - min(values), 4)
        allowed_drop[name] = round(noise[name] + 1 / len(case_ids), 4)

    # 기준값(metrics)은 기준선 run(첫 run)의 통과 비율입니다.
    library_splits[split_name] = {"case_ids": case_ids, "metrics": pass_rates(repeat_reports[0]["cases"], case_ids),
                                  "noise": noise, "allowed_drop": allowed_drop, "absolute_floor": library_absolute_floor}

display(library_splits)

**3단계 · 기준선 dict**: 기준선 run의 ID와 실행 조건, 늘 통과한 사례, 범위별 기준값을 묶습니다. 다음 절에서 이 dict를 파일로 저장합니다.

In [ ]:
library_baseline = {"run_id": repeat_reports[0]["run_id"], "params": top3_params,
                    "stable_cases": library_stable_cases, "splits": library_splits}

print("기준선 칸:", list(library_baseline), "/ 평가 범위:", list(library_baseline["splits"]))

### 3-2. 기준선 run 지정과 파일 저장

승인한 기준선은 두 곳에 남깁니다. 한 곳은 사람이 화면에서 찾아보려고, 다른 한 곳은 CI가 판정에 쓰려고 남깁니다.

| 남기는 곳 | 누가 쓰나 | 무엇을 남기나 |
|---|---|---|
| MLflow run 태그 | 사람(MLflow 화면) | 기준선 run에 `baseline_role=champion` 태그 |
| `baselines/baseline.json` 파일 | CI(pytest) | 판정에 쓰는 기준값과 원래 run의 run_id |

**태그**: champion은 "현재 기준선"이라는 뜻입니다. 기준선 run은 늘 하나여야 하므로, 새 run에 champion을 붙이기 전에 지금 champion인 run(이전 기준선)을 찾아 태그 값을 `retired`(물러난 기준선)로 바꿉니다. 그래서 이 노트북을 다시 실행해 기준선을 새로 정해도 champion은 하나만 남습니다.

**파일**: CI는 실행할 때마다 새 가상 컴퓨터(러너)에서 시작하므로 이전 run이 담긴 `mlflow.db`가 없습니다. 그래서 판정에 쓸 숫자를 저장소의 파일에 두고, 원래 run을 찾을 수 있게 run_id를 함께 적습니다. 파일의 키는 다음과 같습니다.

- `run_id`, `params`: 기준선 run과 그 실행 조건
- `stable_cases`: 늘 통과한 사례
- `splits.full`, `splits.smoke`: 평가 범위마다 `case_ids`(평가한 사례), `metrics`(기준선 지표), `noise`(지표 흔들림), `allowed_drop`(허용 하락폭), `absolute_floor`(절대 하한)

In [ ]:
from mlflow import MlflowClient

# 이미 끝난 run 에 태그를 붙이려면 MlflowClient 의 set_tag(run_id, 키, 값)을 씁니다.
# (mlflow.set_tag 는 지금 열려 있는 run 에만 붙습니다.)
client = MlflowClient()

# 1. 지금 champion(현재 기준선) 태그가 붙은 run 을 찾습니다. 전에 지정한 기준선 run 입니다.
#    처음 실행할 때는 하나도 없고, 노트북을 다시 실행하면 지난번 기준선 run 이 나옵니다.
#    mlflow.search_runs: 실험에서 조건(filter_string)에 맞는 run 을 찾아 DataFrame(한 행이 run 하나)으로 돌려줍니다.
old_champions = mlflow.search_runs(experiment_names=["library-rag-eval"], filter_string="tags.baseline_role = 'champion'")

# 2. 찾은 run 의 baseline_role 태그 값을 champion -> retired(물러난 기준선)로 덮어씁니다.
for run_id in old_champions["run_id"]:
    client.set_tag(run_id, "baseline_role", "retired")

# 3. 이번에 승인한 run 에 champion 을 붙입니다. 이제 champion 은 이 run 하나뿐입니다.
client.set_tag(library_baseline["run_id"], "baseline_role", "champion")
print("기준선 run_id:", library_baseline["run_id"])

# CI 프로젝트 폴더(이 교안 폴더 안의 rag_eval_ci)에 기준선 파일을 저장합니다.
project_dir = material_dir / "교안_02_GitHub_Actions_기초" / "rag_eval_ci"
baseline_path = project_dir / "baselines" / "baseline.json"

baseline_path.parent.mkdir(parents=True, exist_ok=True)
baseline_path.write_text(json.dumps(library_baseline, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

print("저장:", baseline_path)

### 3-3. 기준선 갱신 절차

기준선을 바꾸는 것은 "이제부터 이 정도를 정상으로 본다"는 결정입니다.

1. 품질을 올린 변경, 골든셋 개정, 판사 모델 교체가 있을 때만 새 기준선 후보를 만듭니다. 판사를 바꾸면 같은 답변도 판정이 달라져 이전 기준선과 비교할 수 없습니다.
2. 새 기준선 후보 설정을 반복 평가해 지표 흔들림을 다시 계산합니다.
3. 이전 기준선과 사례 단위로 비교해, 실패로 바뀐 사례가 없는지 사람이 읽고 확인합니다.
4. 새 `baseline.json`을 변경 요청으로 올려 리뷰어 승인을 받은 뒤 병합합니다. 병합할 때 MLflow의 champion 태그도 새 run으로 옮깁니다.

## 4. 변경 요청 평가와 비교

### 4-1. 변경 설정 평가

"응답 비용을 줄이자"는 변경 요청은 `top_k`를 3에서 1로 바꿉니다(변경 설정). 비교하려는 `top_k` 말고는 조건이 모두 같아야 차이의 원인을 알 수 있습니다.

In [ ]:
# evaluate 는 predict_fn 에 question 만 넘기므로, top_k 를 바꾸려면 답변 함수를 따로 감싸야 합니다(기본값은 3).
def answer_top1(question):
    return answer_with_citations(question, top_k=1)


# 비교하려는 top_k 만 다르고 나머지 조건은 현재 설정과 같습니다.
top1_params = {"prompt_version": "v1", "app_model": model_name, "judge_model": eval_model_name, "top_k": 1}

top1_result = evaluate_run("top1", library_records, answer_top1, gate_scorers, top1_params)

### 4-2. `search_runs`로 run 비교

`mlflow.search_runs`는 실험의 run 목록을 표(DataFrame)로 돌려줍니다. run 이름은 `tags.mlflow.runName`, 실행 조건은 `params.이름`, 지표는 `metrics.이름` 열에 들어 있습니다(`correctness/mean`의 `/mean`은 사례 통과 비율이라는 뜻).

`filter_string`으로 원하는 run만 고르고 `order_by`로 정렬합니다. 노트북을 다시 실행하면 같은 이름의 run이 실험에 계속 쌓이므로, 이번 실행에서 만든 네 run(현재 설정 세 run과 변경 설정 run `top1`)만 run_id로 골라 한 표에서 봅니다.

In [ ]:
# 비교할 run 은 현재 설정 세 run(top3_results, 결과 3개)과 변경 설정 run(top1_result, 결과 1개)입니다.
compare_ids = []
for run_result in top3_results:
    compare_ids.append(run_result.run_id)
compare_ids.append(top1_result.run_id)

# filter_string 안의 문자열 값은 따옴표로 감쌉니다. 결과: attributes.run_id IN ('a', 'b', 'c', 'd')
id_text = "', '".join(compare_ids)
runs = mlflow.search_runs(
    experiment_names=["library-rag-eval"],
    filter_string=f"attributes.run_id IN ('{id_text}')",
    order_by=["attributes.start_time ASC"],
)

display(runs[["tags.mlflow.runName", "params.top_k", "metrics.correctness/mean", "metrics.citation_valid/mean", "metrics.case_count"]])

### 4-3. 기준선 run과 사례별 비교

평균만으로는 어떤 사례가 나빠졌는지 알 수 없습니다. 기준선 run(`top3-a`)과 `top1`의 사례별 판정을 `case_id`로 나란히 놓고, 늘 통과한 사례인지도 함께 표시합니다.

In [ ]:
# 두 결과표를 case_id 순으로 정렬해 같은 행이 같은 사례가 되게 합니다.
top3_df = top3_results[0].result_df.sort_values("case_id/value").reset_index(drop=True)
top1_df = top1_result.result_df.sort_values("case_id/value").reset_index(drop=True)
compare = pd.DataFrame({
    "case_id": top3_df["case_id/value"],
    "정답 판정 top3": top3_df["correctness/value"],
    "정답 판정 top1": top1_df["correctness/value"],
    "인용 검증 top3": top3_df["citation_valid/value"],
    "인용 검증 top1": top1_df["citation_valid/value"],
})

# 현재 설정 세 run 에서 모두 통과한 사례인지 표시합니다.
stable_marks = []
for case_id in compare["case_id"]:
    stable_marks.append(case_id in library_baseline["stable_cases"])
compare["늘 통과"] = stable_marks

display(compare)

`top_k=1`은 검색 문서를 하나만 넘기므로 답에 필요한 문서가 검색 1위가 아니면 판정이 `False`로 바뀔 수 있습니다. top3 열과 top1 열의 판정이 다른 행을 봅니다.

- **`늘 통과`가 `True`인데 top1의 두 판정 열 중 하나라도 `False`인 행**: 늘 통과한 사례가 실패한 것이라 사례별 회귀입니다.
- **`늘 통과`가 `False`인 행**: 원래 흔들리거나 늘 실패하던 사례라, 판정이 달라져도 사례별 회귀로 보지 않습니다.

평균 차이보다 "어떤 사례가 실패로 바뀌었는가"가 변경을 받아들일지 정하는 근거가 됩니다. 이 판단을 사람이 표를 읽는 대신 테스트로 자동화하는 것이 pytest 회귀 테스트입니다.

## 5. pytest로 판정하기

### 5-1. 평가 결과를 파일로 남기기

평가는 LLM을 호출하므로 느리고 비용이 듭니다. 그래서 **평가는 한 번 실행해 결과를 파일로 남기고, pytest는 그 파일만 읽어 판정**하게 나눕니다. 판정 규칙만 고칠 때는 평가를 다시 돌리지 않아도 됩니다.

평가 결과는 MLflow에도 있지만, run 지표에는 채점 함수별 평균만 있고 사례별 판정과 채점 오류 수는 사례별 기록(trace)에 흩어져 있습니다. 그래서 `make_report`로 이 둘을 한 파일에 모아, 테스트가 MLflow를 조회하지 않고 바로 읽게 합니다. 파일에는 `run_id`도 남기므로 자세한 내용은 그 run을 MLflow 화면에서 찾아봅니다.

<img src="../images/09_eval_and_gate_files.png" width="1050" alt="평가 실행은 LLM을 호출해 느리고 결과를 reports/results.json 에 남기며, pytest 판정은 이 파일과 사람이 승인한 baselines/baseline.json 만 읽어 빠르게 종료 코드 0(통과) 또는 1(실패)을 내고, CI가 이 코드로 검사 결과를 표시합니다.">

먼저 테스트 파일이 제대로 동작하는지 보려고, 2-2에서 같은 설정으로 평가한 둘째 run(`top3-b`)의 결과를 `reports/results.json`으로 저장합니다. 이 run은 지표 흔들림 계산에 쓴 run이라 허용 하락폭과 사례별 회귀 테스트는 반드시 통과합니다. 절대 하한 테스트와 채점 오류 테스트(`test_no_scoring_errors`)만 이 run 결과에 따라 통과하거나 실패합니다.

In [ ]:
reports_dir = project_dir / "reports"
reports_dir.mkdir(parents=True, exist_ok=True)

results_path = reports_dir / "results.json"
results_path.write_text(json.dumps(repeat_reports[1], ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

print("저장:", results_path, "/ 지표:", repeat_reports[1]["metrics"])

### 5-2. 테스트 파일 읽기

pytest는 `test_`로 시작하는 파일의 `test_` 함수를 실행하고, 함수 안의 `assert` 조건이 거짓이면 그 테스트를 실패로 셉니다. `@pytest.mark.parametrize`는 같은 테스트를 값마다 따로 실행하므로, 사례별 회귀 테스트가 실패하면 실패한 `case_id`가 바로 보입니다.

pytest의 기본 문법은 부록 01에서 자세히 다룹니다.

테스트 파일은 `rag_eval_ci/tests/test_rag_regression.py`에 들어 있습니다. 파일 맨 위에서 이번 평가 결과(`reports/results.json`)와 기준선(`baselines/baseline.json`)을 읽고, 네 규칙을 테스트 함수로 하나씩 확인합니다.

| 규칙 | 테스트 함수 | 확인하는 것 | 실패하는 경우 |
|---|---|---|---|
| 평가 완전성 | `test_same_cases` | 기준선과 같은 사례들로 평가했는가 | 사례가 빠지거나 다른 사례가 섞임 |
| 평가 완전성 | `test_no_scoring_errors` | 채점 오류가 난 칸이 없는가 | 판정이 빈 칸이 하나라도 있음 |
| 절대 하한 | `test_metric_floor` | 채점 함수마다 지표가 `absolute_floor` 이상인가 | 지표가 팀이 정한 최소 품질보다 낮음 |
| 허용 하락폭 | `test_metric_drop` | 채점 함수마다 기준선 대비 하락폭이 `allowed_drop` 이하인가 | 지표가 기준선보다 허용 하락폭을 넘게 떨어짐 |
| 사례별 회귀 | `test_stable_case_still_passes` | 늘 통과한 사례가 이번에도 모든 채점 함수에서 통과했는가 | 늘 통과한 사례 하나가 실패 |

`test_metric_floor`와 `test_metric_drop`은 채점 함수마다, `test_stable_case_still_passes`는 늘 통과한 사례마다 따로 실행되므로, 실패 목록에서 어느 채점 함수나 어느 사례가 걸렸는지 바로 보입니다.


```python
"""reports/results.json(이번 평가 결과)을 baselines/baseline.json(승인한 기준선)과 비교하는 RAG 회귀 테스트입니다.
하나라도 실패하면 종료 코드 1이 되어 CI 검사가 실패합니다.
    평가 완전성: test_same_cases, test_no_scoring_errors
    절대 하한: test_metric_floor
    허용 하락폭: test_metric_drop
    사례별 회귀: test_stable_case_still_passes
"""
import json
from pathlib import Path

import pytest

# 이번 평가 결과(report)와 승인한 기준선(baseline)을 읽습니다.
ROOT = Path(__file__).resolve().parents[1]
report = json.loads((ROOT / "reports" / "results.json").read_text(encoding="utf-8"))
baseline = json.loads((ROOT / "baselines" / "baseline.json").read_text(encoding="utf-8"))
# 이번 평가 범위(smoke 또는 full)의 기준값과, 판정할 채점 함수 이름 목록
split = baseline["splits"][report["split"]]
JUDGED_SCORERS = list(split["metrics"])

# 사례별 회귀는 늘 통과하던 사례 중 이번에 평가한 것만 봅니다.
stable_in_scope = []
for case_id in baseline["stable_cases"]:
    if case_id in report["cases"]:
        stable_in_scope.append(case_id)


def test_same_cases():
    # 평가 완전성: 기준선과 같은 사례로 평가해야 지표를 비교할 수 있습니다(순서는 무시하려고 정렬).
    assert sorted(report["cases"].keys()) == sorted(split["case_ids"]), "기준선과 다른 사례로 평가했습니다"


def test_no_scoring_errors():
    # 평가 완전성: 채점 오류가 난 칸은 판정이 비어 있어 지표를 믿을 수 없습니다.
    assert report["errors"] == 0, f"채점 오류 {report['errors']}건"


# parametrize: 채점 함수마다 테스트를 따로 실행합니다.
@pytest.mark.parametrize("name", JUDGED_SCORERS)
def test_metric_floor(name):
    # 절대 하한: 이번 지표가 팀이 정한 최소 품질 이상이어야 합니다.
    current = report["metrics"][name]
    assert current >= split["absolute_floor"][name], f"{name} {current} 가 절대 하한 {split['absolute_floor'][name]} 보다 낮습니다"


@pytest.mark.parametrize("name", JUDGED_SCORERS)
def test_metric_drop(name):
    # 허용 하락폭: 기준선 대비 하락폭이 허용값 이하여야 합니다(오르면 음수라 통과, 소수 오차는 반올림으로 제거).
    drop = round(split["metrics"][name] - report["metrics"][name], 4)
    assert drop <= split["allowed_drop"][name], f"{name} 가 기준선보다 {drop} 떨어졌습니다(허용 {split['allowed_drop'][name]})"


# parametrize: 사례마다 따로 실행해 실패한 case_id 가 바로 보입니다.
@pytest.mark.parametrize("case_id", stable_in_scope)
def test_stable_case_still_passes(case_id):
    # 사례별 회귀: 평균이 허용 범위 안이어도, 늘 통과하던 사례가 새로 실패하면 잡습니다.
    case = report["cases"][case_id]
    for name in JUDGED_SCORERS:
        assert case[name] is True, f"{case_id}: 기준선에서 늘 통과하던 질문이 {name} 에서 실패했습니다"
```

### 5-3. pytest 실행과 종료 코드

터미널에서는 저장소 최상위 폴더에서 이 교안 폴더 안의 CI 프로젝트 폴더로 이동해 실행하며, `-q`는 결과를 짧게 보여 주는 옵션입니다. `rag_eval_ci`는 자기 `pyproject.toml`이 있는 별도 프로젝트라, 이 폴더에서 처음 `uv run`을 하면 전용 환경을 먼저 설치합니다.

```bash
cd day61_평가자동화_MLflow/교안_02_GitHub_Actions_기초/rag_eval_ci
uv run pytest -q tests
```

노트북에서는 pytest를 `subprocess.run`으로 실행하고 **종료 코드**(exit code, `returncode`, 프로그램이 끝나며 돌려주는 숫자)를 봅니다.

- **0**: 모든 테스트 통과
- **1**: 하나 이상 실패
- **2**: 결과 파일이 없는 등의 이유로 테스트 수집(collection) 중 오류

CI는 pytest 종료 코드가 0이 아니면 그 단계(step)를 실패로 표시합니다. 저장소에서 이 결과를 필수 검사(통과해야 병합할 수 있는 검사)로 지정하면 병합이 막힙니다.

In [ ]:
import subprocess
import sys


def run_pytest(folder):
    """폴더의 tests 를 pytest 로 실행하고 종료 코드와 결과를 출력합니다."""
    # sys.executable 로 실행해야 노트북과 같은 가상환경의 pytest 가 실행됩니다.
    # -vv 는 테스트마다 PASSED/FAILED 를 한 줄씩 보여 주고 실패 메시지를 자르지 않습니다. --tb=no 는 실패한 코드 위치 출력을 생략합니다.
    # --color=no 는 출력에 색 코드가 섞이지 않게, -p no:deepeval 은 설치된 DeepEval 의 pytest 플러그인을 불러오지 않게 합니다.
    command = [sys.executable, "-m", "pytest", "-vv", "--tb=no", "--color=no", "-p", "no:deepeval", "tests"]
    completed = subprocess.run(command, cwd=folder, capture_output=True, text=True)
    print("종료 코드:", completed.returncode)
    print(completed.stdout)


run_pytest(project_dir)

이번에는 변경 설정(`top_k=1`)의 결과를 같은 `results.json`에 덮어쓰고 pytest를 다시 실행합니다.

In [ ]:
# 평가 결과를 테스트가 읽는 형식(지표, 사례별 판정, 오류 수)으로 바꿉니다.
top1_report = make_report(top1_result, "full")
# pytest 는 별도 프로세스라 노트북 변수를 못 보므로, 같은 결과 파일에 덮어씁니다.
results_path.write_text(json.dumps(top1_report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("저장한 지표:", top1_report["metrics"])

run_pytest(project_dir)

두 pytest 실행의 `종료 코드:` 줄과 마지막 줄(`N passed`, `N failed`)을 비교합니다. 하락이 있는 실행은 `FAILED` 줄에 실패한 테스트 이름과 `assert` 메시지가 함께 나와 어느 규칙에서 왜 실패했는지 읽을 수 있습니다.

사례별 회귀 테스트에서 실패하는 `case_id`는 4-3의 비교표에서 `늘 통과`가 `True`인데 top1의 판정 열 중 하나라도 `False`인 사례입니다.

### 5-4. 비교: DeepEval `assert_test`

이 절은 비교용입니다. 회귀 게이트는 5-3까지의 방식(평가 한 번, 결과 파일, pytest 판정)을 그대로 쓰고, 여기서는 테스트 안에서 판사를 부르는 DeepEval 방식과 쓰임새만 비교합니다.

DeepEval은 지표 판정을 pytest 테스트로 만드는 `assert_test`를 제공합니다. 테스트 파일에 아래처럼 쓰고 `deepeval test run tests/test_quality.py`로 실행하면, 테스트마다 판사가 점수를 매기고 통과 점수(`threshold`, 여기서는 0.7)보다 낮으면 `AssertionError`로 실패합니다.

```python
from deepeval import assert_test

def test_answer_matches_expected_output():
    # test_case(LLMTestCase)와 correctness_metric(GEval)은 아래 셀과 같은 방식으로 만듭니다.
    # assert_test 한 번은 사례 하나를 평가합니다. 여러 사례는 @pytest.mark.parametrize 로 사례마다 테스트를 만듭니다.
    assert_test(test_case, [correctness_metric])
```

이 교안에서 만든 방식과는 쓰임새가 다릅니다.

| 방식 | 판정 시점 | 맞는 자리 |
|---|---|---|
| MLflow 평가 결과 파일 + pytest | 평가를 한 번 돌린 뒤 파일만 판정 | 기준선과 비교하고 평가 run 기록이 남아야 하는 회귀 게이트 |
| DeepEval `assert_test` | 테스트 실행 중에 판사 호출 | 사례마다 통과 점수를 적용할 때 |

아래 셀은 `top_k=1`에서 정답 판정이 `False`였던 첫 사례를 같은 판단 단계(`CORRECTNESS_STEPS`)와 판사(`judge_llm`)로 `measure()`해, `assert_test`가 테스트 안에서 하는 판정을 직접 보여 줍니다.

In [ ]:
# case_id 순으로 정렬해 두어야 다시 실행해도 같은 사례가 골라집니다.
top1_df = top1_result.result_df.sort_values("case_id/value")
failed_row = top1_df[top1_df["correctness/value"] == False].iloc[0]

# 같은 case_id 의 골든 사례에서 질문과 정답 답변을 찾습니다.
for case in library_golden:
    if case["case_id"] == failed_row["case_id/value"]:
        failed_case = case

# 정답 판정 채점 함수와 같은 G-Eval 입니다. assert_test 에 넘기면 이 판정이 테스트가 됩니다.
correctness_metric = GEval(
    name="정답 판정",
    evaluation_steps=CORRECTNESS_STEPS,
    evaluation_params=[SingleTurnParams.INPUT, SingleTurnParams.ACTUAL_OUTPUT, SingleTurnParams.EXPECTED_OUTPUT],
    model=judge_llm,
    threshold=0.7,
    async_mode=False,  # 정답 판정 채점 함수와 같은 설정입니다.
)

# response 열에는 답변 함수가 돌려준 dict 가 들어 있어 answer 를 꺼내 씁니다.
test_case = LLMTestCase(input=failed_case["input"], actual_output=failed_row["response"]["answer"],
                        expected_output=failed_case["expected_output"])
correctness_metric.measure(test_case)

print("사례:", failed_case["case_id"], "/ 질문:", failed_case["input"])
print("점수:", round(correctness_metric.score, 2), "/ 통과:", correctness_metric.success)
print("이유:", correctness_metric.reason)

`통과: False`이면 같은 사례를 `assert_test`로 검사하는 테스트도 `AssertionError`로 실패합니다.

## 6. CI 회귀 게이트

지금까지는 노트북에서 평가와 pytest를 직접 실행했습니다. 이제 같은 평가와 테스트를 변경 요청마다 CI가 자동으로 돌리게 합니다.

### 6-1. CI에서 돌릴 프로젝트 구조

이 교안 폴더 안의 `rag_eval_ci/`는 그대로 저장소 하나가 되도록 구성했습니다. 이 폴더를 GitHub 저장소로 올리면 아래에서 읽을 워크플로 파일이 동작합니다.

```text
rag_eval_ci/
├── app.py              도서관 안내 RAG 앱(검색 문서 수 TOP_K, 프롬프트). 변경 요청은 보통 이 파일을 고칩니다
├── run_eval.py         골든셋으로 app.py 를 평가해 reports/ 에 결과를 남깁니다
├── data/               문서, 골든셋, 문서 벡터
├── baselines/          baseline.json (기준선 파일, 노트북이 저장)
├── tests/              test_rag_regression.py (5-2에서 읽은 테스트 파일)
├── pyproject.toml, uv.lock   고정한 라이브러리 버전
├── .gitignore          reports/ 처럼 실행할 때마다 새로 생기는 파일은 저장소에 올리지 않습니다
└── .github/workflows/rag-eval.yml   (아래에서 읽는 GitHub Actions 워크플로)
```

`run_eval.py`는 이 교안과 같은 채점 함수와 `make_report`로 평가해 `reports/results.json`을 쓰고, CI 화면에 붙일 요약 `reports/summary.md`도 만듭니다. 평가만 하고 통과 여부 판정은 pytest에 맡깁니다.

`--split smoke`는 골든셋에서 `smoke` 표시가 있는 대표 6건만, `--split full`은 16건 전부 평가합니다. 로컬에서 CI와 같은 순서로 실행해 볼 수 있으며, API 비용이 들므로 smoke 6건만 평가합니다.

```bash
cd day61_평가자동화_MLflow/교안_02_GitHub_Actions_기초/rag_eval_ci
uv sync
uv run python run_eval.py --split smoke
uv run pytest -q tests
```

### 6-2. GitHub Actions 워크플로

GitHub Actions는 저장소의 `.github/workflows/` 폴더에 있는 YAML(설정 파일 형식) 파일을 읽습니다. 이 파일 하나를 워크플로(workflow)라고 하며, 여기서는 `rag_eval_ci/.github/workflows/rag-eval.yml`입니다. 워크플로는 다음 세 가지를 정합니다.

1. **언제 돌지(`on`)**: 변경 요청(`pull_request`)이 앱·데이터·테스트 파일을 바꿀 때, 매일 새벽 예약 실행(`schedule`), 화면에서 직접 실행(`workflow_dispatch`)
2. **어디서 돌지(`runs-on`)**: GitHub가 제공하는 우분투 가상 컴퓨터(러너)
3. **무엇을 할지(`jobs` > `steps`)**: 작업(job) `evaluate` 하나가 단계(step)를 차례로 실행합니다. 저장소 받기, uv 설치, 라이브러리 설치, 평가, 테스트, 결과 요약과 산출물(artifact, 보관용 결과 파일) 저장

변경 요청에서는 비용을 줄이려고 smoke 6건만, 예약·직접 실행에서는 full 16건을 평가합니다. API 키는 YAML에 쓰지 않고 저장소 설정의 비밀 변수(**Secrets**)에 `OPENAI_API_KEY`로 등록한 뒤 `${{ secrets.OPENAI_API_KEY }}`를 작업의 환경 변수(`env`)로 넘깁니다.


```yaml
name: rag-eval

on:
  pull_request:
    paths:
      - "app.py"
      - "run_eval.py"
      - "data/**"
      - "tests/**"
      - "baselines/**"
      - "pyproject.toml"
      - "uv.lock"
      - ".github/workflows/rag-eval.yml"
  schedule:
    - cron: "0 17 * * *"   # 매일 UTC 17시(한국 시간 새벽 2시)에 전체 사례 평가
  workflow_dispatch:

permissions:
  contents: read

# 같은 브랜치에 새 커밋이 올라오면 진행 중인 이전 평가를 취소해 비용을 아낍니다.
concurrency:
  group: rag-eval-${{ github.ref }}
  cancel-in-progress: true

jobs:
  evaluate:
    runs-on: ubuntu-latest
    timeout-minutes: 20
    env:
      OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
    steps:
      - uses: actions/checkout@v7
      - uses: astral-sh/setup-uv@v10.2.0
        with:
          python-version: "3.12"
          enable-cache: true
      - name: 라이브러리 설치
        run: uv sync --locked
      - name: 평가 실행
        run: |
          if [ "${{ github.event_name }}" = "pull_request" ]; then
            uv run python run_eval.py --split smoke
          else
            uv run python run_eval.py --split full
          fi
      - name: 회귀 테스트
        run: uv run pytest -q tests --junitxml=reports/junit.xml
      - name: 결과 요약
        if: always()
        run: |
          if [ -f reports/summary.md ]; then
            cat reports/summary.md >> "$GITHUB_STEP_SUMMARY"
          fi
      - name: 평가 산출물 저장
        if: always()
        uses: actions/upload-artifact@v7
        with:
          name: rag-eval-reports
          path: reports/
          retention-days: 14
```

| 키 | 뜻 |
|---|---|
| `paths` | 이 경로의 파일이 바뀐 변경 요청에서만 실행합니다. README만 고친 변경 요청은 평가하지 않습니다 |
| `cron: "0 17 * * *"` | 다섯 칸은 분, 시, 일, 월, 요일 순서입니다. 시각은 UTC(세계 표준시) 기준이며 한국 시간은 UTC + 9시간입니다 |
| `uv sync --locked` | `uv.lock`에 적힌 버전 그대로 설치하고, `uv.lock`이 `pyproject.toml`과 어긋나 갱신이 필요하면 오류로 중단합니다. 로컬과 같은 버전으로 평가하기 위해서입니다 |
| `--junitxml` | 테스트 결과를 JUnit 형식의 XML 파일로도 남겨 산출물에 넣습니다 |
| `if: always()` | 앞 단계가 실패해도 요약과 산출물 저장은 실행합니다. 실패했을 때 결과를 봐야 하기 때문입니다 |
| `$GITHUB_STEP_SUMMARY` | 이 변수가 가리키는 파일에 쓴 마크다운이 실행 화면의 요약(Summary)에 표시됩니다 |

이벤트(`on`), `if`, 권한·시간 제한·동시 실행 같은 문법은 부록 02에서 자세히 다룹니다.

### 6-3. 실제 저장소에서 돌려 본 결과

`rag_eval_ci/`를 GitHub 비공개 저장소로 올리고 `OPENAI_API_KEY`를 Secrets에 등록한 뒤, 변경 요청 두 개를 올려 워크플로를 실제로 돌렸습니다.

- **변경 요청 1**: `app.py`의 설정 주석만 보완했습니다.
- **변경 요청 2**: 응답 비용을 줄이려고 검색 문서 수를 3개에서 1개로 줄였습니다(`TOP_K = 3 -> 1`).

실행 당시 저장소에 올린 기준선 파일로 판정했으므로, 여러분이 만든 기준선과는 늘 통과한 사례 수가 달라 테스트 개수가 다를 수 있습니다. 저장소의 **Actions** 탭에는 워크플로 실행(workflow run)마다 한 줄씩 결과가 쌓이며, 빨간 X는 실패, 초록 체크는 성공입니다.

<img src="../images/gha_01_runs.png" width="1050" alt="GitHub Actions 실행 목록. 검색 문서 수를 1개로 줄인 변경 요청 2의 실행은 실패(빨간 X), 주석만 보완한 변경 요청 1의 실행은 성공(초록 체크)으로 표시됩니다.">

- 누를 곳: 저장소 위쪽 Actions 탭 > Event 필터에서 `pull_request`
- 주소: `https://github.com/<계정>/<저장소>/actions?query=event%3Apull_request`

변경 요청 1의 실행을 열고 `evaluate` 작업에서 **회귀 테스트** 단계를 펼치면 pytest 출력이 보입니다. "평가 실행" 단계는 로그에 `{"correctness": 0.8333, "citation_valid": 1.0}`을 출력했고, 기준선과 비교한 테스트 11개가 모두 통과했습니다.

<img src="../images/gha_04_passed_job.png" width="1050" alt="변경 요청 1 실행의 회귀 테스트 단계. uv run pytest 명령이 돌고 11 passed가 출력됩니다.">

- 누를 곳: Actions > 실행 이름(앱 설정 주석 보완) > 왼쪽 evaluate > 회귀 테스트 단계 펼치기
- 주소: `https://github.com/<계정>/<저장소>/actions/runs/<실행 ID>/job/<작업 ID>`

변경 요청 2의 실행은 "평가 실행" 단계에서 로그에 `{"correctness": 0.5, "citation_valid": 1.0}`을 출력했고, 회귀 테스트 단계에서 4개가 실패했습니다.

GitHub는 실패한 단계를 자동으로 펼쳐 로그 끝을 보여 주고, `short test summary info` 아래에 실패한 테스트와 테스트 파일의 `assert` 메시지가 그대로 나옵니다. 마지막 줄의 종료 코드가 1이므로 이 단계와 실행 전체가 실패로 표시됩니다.

<img src="../images/gha_02_failed_job.png" width="1050" alt="변경 요청 2 실행의 회귀 테스트 단계. test_metric_floor, test_metric_drop, lib-09와 lib-10의 test_stable_case_still_passes가 FAILED로 나오고 4 failed, 7 passed와 exit code 1이 보입니다.">

- 누를 곳: Actions > 실행 이름(검색 문서 수를 1개로 줄여 응답 비용 절감) > 왼쪽 evaluate
- 주소: `https://github.com/<계정>/<저장소>/actions/runs/<실행 ID>/job/<작업 ID>`

- **정답 판정**: smoke 6건 가운데 3건(0.5)만 통과해 절대 하한, 허용 하락폭, 사례별 회귀 세 규칙에서 모두 실패했습니다.
- **인용 검증**: 1.0으로 기준선 값과 같습니다. 검색한 문서 한 개를 그대로 인용했으므로 인용은 맞고, 답의 내용만 틀린 회귀입니다.

판정이 흔들리므로 같은 변경이라도 실패하는 규칙은 실행마다 다를 수 있습니다. 변경 요청 화면에서도 결과가 보이며, Conversation 탭 맨 아래 검사 영역에 `rag-eval / evaluate (pull_request)`(워크플로 이름 / 작업 이름 (이벤트))가 실패로 표시됩니다.

<img src="../images/gha_03_pr_checks.png" width="1050" alt="변경 요청 2 화면 아래쪽의 검사 영역. Some checks were not successful 아래에 rag-eval / evaluate (pull_request) 검사가 실패로 표시됩니다.">

- 누를 곳: 저장소 위쪽 Pull requests 탭 > 변경 요청 2 > Conversation 탭 맨 아래
- 주소: `https://github.com/<계정>/<저장소>/pull/<번호>`

검사가 실패해도 지금 상태에서는 병합 버튼을 누를 수 있습니다. 실패한 검사로 병합을 막는 Rulesets 설정과 요금제별 적용 조건은 부록 02에서 다룹니다.

이 워크플로는 `paths`를 지정했으므로, 필수 검사로 만들기 전에 실무에서 지킬 것의 `paths` 항목을 함께 봅니다.

실행 화면의 요약(Summary)에는 `run_eval.py`가 만든 `summary.md`가 붙습니다. 변경 요청 2의 요약은 다음과 같았습니다(같은 내용이 산출물 `rag-eval-reports`에 `junit.xml`, `results.json`, `mlflow.db`와 함께 들어 있습니다).

| 사례 | 정답 판정 | 인용 검증 |
|---|---|---|
| lib-01 | 통과 | 통과 |
| lib-04 | 통과 | 통과 |
| lib-09 | 실패 | 통과 |
| lib-10 | 실패 | 통과 |
| lib-13 | 실패 | 통과 |
| lib-15 | 통과 | 통과 |

`lib-13`은 기준선에서도 늘 실패하는 사례라 사례별 회귀 테스트에는 들어가지 않고, 평균 지표에만 반영됩니다.

예외 조건 유형의 대표로 smoke에 넣은 이 사례가 늘 실패하므로 smoke의 정답 판정 기준선 값은 많아야 5/6(0.8333)입니다. 한 건이 더 실패해도 4/6(0.6667)이라 절대 하한 0.6 이상이지만, 두 건이 더 실패하면 3/6(0.5)이 되어 절대 하한 테스트에서 실패합니다.

smoke가 실패하면 실패한 사례와 이유를 먼저 읽고, 흔들리던 사례 때문이면 16건 전체 평가(full)로 다시 확인합니다. 실무에서는 늘 실패하는 사례를 smoke 대표에서 빼 두기도 합니다.

### 6-4. 실무에서 지킬 것

- **비밀 변수가 없는 실행과 멈춘 실행에 대비합니다.** fork(복제 저장소)에서 온 변경 요청에는 비밀 변수가 전달되지 않습니다([GitHub 비밀 변수 문서](https://docs.github.com/en/actions/how-tos/write-workflows/choose-what-workflows-do/use-secrets)). 공개 저장소라면 키가 필요 없는 테스트와 평가를 나누고, 평가는 관리자가 승인한 실행에서만 돌립니다. `timeout-minutes`를 두면 API 응답을 기다리며 멈춘 실행도 그 시간에 끝납니다.
- **필수 검사인 워크플로에 `paths`를 두면 병합이 막힐 수 있습니다.** 바뀐 파일이 `paths`에 없어 워크플로가 아예 돌지 않으면 필수 검사가 "대기 중"으로 남으므로, 필수 검사로 지정할 때는 `paths`를 빼거나 워크플로 안에서 변경 파일을 확인해 평가만 건너뜁니다. ([필수 검사 문제 해결](https://docs.github.com/en/pull-requests/how-tos/merge-and-close-pull-requests/troubleshooting-required-status-checks))
- **테스트 실패를 성공으로 바꾸는 설정을 쓰지 않습니다.** `|| true`나 `continue-on-error: true`를 붙이면 테스트가 실패해도 단계가 통과로 표시됩니다.
- **늘 통과한 사례도 가끔 한 번 실패합니다.** 반복 평가 몇 번으로 고른 사례에도 드문 실수(예: 모델이 근거 ID를 `interlibrary- loan`처럼 공백을 넣어 적음)가 남아, 아무것도 바꾸지 않은 변경 요청이 사례별 회귀 테스트에서 실패할 수 있습니다.
- **막기 전에 되풀이되는지 봅니다.** 실패 이유를 읽고 그 사례만 다시 평가해 같은 실패가 되풀이되는지 확인한 뒤 병합을 막을지 정합니다. 되풀이되지 않는 실패가 잦으면 반복 횟수를 늘려 지표 흔들림을 다시 계산합니다.

## 7. 🖐️ 함께 따라하기: HR 상담 챗봇의 프롬프트 개정 판정하기

본 실습의 반복 평가, 기준선, 테스트 파일을 다른 챗봇의 다른 종류의 변경에 그대로 적용합니다. CI는 같은 테스트 파일을 실행할 뿐이라 따라하기에서는 노트북에서 pytest까지만 돌립니다.

교안 01의 따라하기에서 평가한 HR 상담 챗봇(고용노동부 「2024 유연근무 활용 매뉴얼」)에 회귀 게이트를 적용합니다. 상황은 다음과 같습니다.

1. 직원들이 "인용된 문장이 딱딱해서 읽기 어렵다"는 의견을 냈습니다.
2. 담당자가 프롬프트의 "구절을 바꿔 쓰거나 요약하지 마세요"를 "**인용 구절은 읽기 쉽게 다듬어 쓰세요**"로 바꾸는 변경 요청을 올렸습니다. 바로 앞의 "글자 그대로 복사한 구절을 인용하세요"는 그대로 남아 있어, 두 지시가 부딪칠 때 모델이 어느 쪽을 따르는지는 평가로 확인해야 합니다.
3. 질문 몇 개를 넣어 보니 답의 내용은 여전히 맞습니다.
4. 이 변경을 받아들여도 되는지 기준선과 회귀 테스트로 판정합니다.

**실습 진행 방법**

- **직접 설계:** 클래스·함수 이름과 입출력 규약을 보고 핵심 로직 전체를 작성합니다.
- **부분 작성:** 제공한 보조 코드의 `...`를 채웁니다. 환경 준비·데이터·긴 답변 프롬프트·결과 출력은 활용하세요.
- 작성 셀을 완성한 뒤 아래 실행·출력 셀로 확인합니다. `...`는 실행 가능한 기본값이 아닙니다.
- 완성 코드는 수업이 끝난 뒤 강사가 공유합니다. 먼저 자신의 구현으로 각 절을 완성해 봅니다.

본 실습의 `evaluate_run`, `make_report`, `pass_rates`, `run_pytest`와 테스트 파일을 그대로 쓰고, 기준선은 본 실습과 같은 세 단계로 만듭니다. 결과 파일은 이 교안 폴더 안의 `hr_eval_ci/`에 둡니다.

### 7-1. HR 상담 챗봇 부품 준비

아래 네 셀은 교안 01 따라하기와 같은 일을 하는 코드라 차례로 실행만 하면 됩니다. 인용 검증 채점 함수만 `@scorer(name="citation_valid")`로 이름을 본 실습과 같게 두어, 본 실습의 `make_report`와 테스트 파일을 고치지 않고 씁니다.

**매뉴얼과 골든셋**: 매뉴얼 문서와 골든 사례를 읽고, 평가 run을 `hr-rag-eval` 실험에 기록하게 합니다.

In [ ]:
hr_docs = read_jsonl("따라하기/docs.jsonl")
hr_golden = read_jsonl("따라하기/golden.jsonl")

hr_texts = {}
for doc in hr_docs:
    hr_texts[doc["doc_id"]] = doc["text"]

hr_store = Chroma(collection_name="hr_manual", embedding_function=embedding_model,
                  persist_directory=str(data_dir / "따라하기" / "chroma"))

hr_records = to_records(hr_golden)
hr_experiment = mlflow.set_experiment("hr-rag-eval")

print("매뉴얼 문서:", len(hr_docs), "/ 골든 사례:", len(hr_records), "/ 실험:", hr_experiment.name)

**매뉴얼 검색**: 질문과 가까운 매뉴얼 문서를 찾습니다.

In [ ]:
@mlflow.trace(span_type="RETRIEVER")
def search_manual(search_query, top_k=4):
    """매뉴얼에서 질문과 가까운 절 top_k 개를 반환합니다."""
    return hr_store.similarity_search(search_query, k=top_k)

**인용 답변 함수**: 지금 서비스 중인 프롬프트(`HR_SYSTEM_V1`)로 답하고 인용을 붙입니다.

In [ ]:
HR_SYSTEM_V1 = ("당신은 회사 인사팀 상담 챗봇입니다. 고용노동부 유연근무 활용 매뉴얼 발췌로만 답하세요. "
                "답변을 부분(segments)으로 나눠 쓰고, 매뉴얼에서 가져온 사실을 쓴 부분마다 근거 ID와 그 절에서 글자 그대로 복사한 구절을 인용하세요. "
                "구절을 바꿔 쓰거나 요약하지 마세요. 적용 대상, 요건, 기간, 예외를 빠뜨리지 마세요. "
                "매뉴얼에 답이 없으면 '제공된 매뉴얼에서 확인할 수 없습니다.'라는 부분 하나만 인용 없이 쓰세요.")
hr_prompt = ChatPromptTemplate.from_messages([("system", HR_SYSTEM_V1), ("human", "질문: {question}\n\n매뉴얼 발췌:\n{context}")])

hr_chain = hr_prompt | llm.with_structured_output(CitedAnswer)


@mlflow.trace
def hr_answer(question):
    """매뉴얼을 검색해 hr_chain 으로 답하고, answer_with_citations 와 같은 dict(segments, answer, searched_ids)를 돌려줍니다."""
    docs = search_manual(question)
    result = hr_chain.invoke({"question": question, "context": format_documents(docs)})

    output = result.model_dump()

    texts = []
    for segment in output["segments"]:
        texts.append(segment["text"])
    output["answer"] = " ".join(texts)

    output["searched_ids"] = []
    for doc in docs:
        output["searched_ids"].append(doc.metadata["source_id"])
    return output

**채점 함수와 실행 조건**: 판정에 쓰는 채점 함수 목록과 run에 남길 조건입니다.

In [ ]:
# 보고서와 테스트가 같은 이름을 읽도록 채점 함수 이름을 citation_valid 로 둡니다.
@scorer(name="citation_valid")
def hr_citation_valid(outputs):
    """인용 검증(HR): citation_valid 와 같은 규칙으로, 원문 사전만 hr_texts 를 써서 판정합니다. 기준선과 같은 이름으로 기록되게 citation_valid 로 이름을 붙입니다."""
    problems = []
    for segment in outputs["segments"]:
        for citation in segment["citations"]:
            status = check_citation(citation, hr_texts, outputs["searched_ids"])
            if status != "확인":
                problems.append(f"{status}: {citation['evidence_id']}")

    if len(problems) == 0:
        return Feedback(value=True, rationale="붙은 인용이 모두 근거 ID·문구 대조를 통과함")
    return Feedback(value=False, rationale="; ".join(problems))


hr_gate_scorers = [correctness, hr_citation_valid]

hr_params = {"prompt_version": "v1", "app_model": model_name, "judge_model": eval_model_name, "top_k": 4}

### 7-2. 현재 프롬프트 반복 평가

**직접 설계 · 같은 설정 두 번 평가**

- `evaluate_run`과 `hr_records`, `hr_answer`, `hr_gate_scorers`, `hr_params`로 현재 프롬프트를 `hr-v1-a`, `hr-v1-b` 두 run으로 평가합니다(사례 24건씩). 사례가 많아 비용을 줄이려고 두 번만 평가합니다.
- 각 결과를 `make_report(결과, "full")`를 써서 보고서로 바꿔 `hr_repeat_reports` 목록에 차례로 담습니다. 첫 번째가 승인할 기준선 run입니다.

In [ ]:
# hr_repeat_reports 를 만드세요.
hr_repeat_reports = ...

for report in hr_repeat_reports:
    print("지표:", report["metrics"])

### 7-3. 기준선 지정과 저장

**직접 설계 · 기준선 파일과 champion 태그**

1. **늘 통과한 사례**: 본 실습 1단계처럼 `hr_repeat_reports`의 모든 run에서 두 채점 함수가 모두 `True`인 사례를 `hr_stable_cases`에 모읍니다.
2. **평가 범위마다 기준값**: `smoke`가 참인 사례를 `hr_smoke_ids`에 모으고, 본 실습 2단계처럼 `full`·`smoke`마다 기준값을 계산해 `hr_splits`에 담습니다. 절대 하한은 `hr_absolute_floor = {"correctness": 0.8, "citation_valid": 0.7}`입니다(팀이 정하는 값입니다. 이 따라하기에서는 인용 검증에 여유를 더 두어 0.7로 둡니다).
3. **기준선 dict**: 기준선 run(`hr_repeat_reports[0]`)의 `run_id`, `hr_params`, `hr_stable_cases`, `hr_splits`를 묶어 `hr_baseline`을 만듭니다.
4. 본 실습에서 만든 `client`(`MlflowClient`)로 `hr-rag-eval` 실험에서 이전 기준선 run의 태그를 `retired`로 바꾸고, 새 기준선 run에 `baseline_role=champion`을 붙입니다.
5. `hr_dir / "baselines" / "baseline.json"`에 저장합니다(`hr_dir`은 코드에 정해 두었습니다).

In [ ]:
# 기준선 세 단계(hr_stable_cases, hr_splits, hr_baseline)와 champion 태그, 파일 저장까지 작성하세요.
hr_stable_cases = ...
hr_smoke_ids = ...
hr_absolute_floor = {"correctness": 0.8, "citation_valid": 0.7}
hr_splits = ...
hr_baseline = ...
hr_dir = material_dir / "교안_02_GitHub_Actions_기초" / "hr_eval_ci"
...

print("늘 통과한 사례:", hr_baseline["stable_cases"])
print("full 기준선 값:", hr_baseline["splits"]["full"])

### 7-4. 개정 프롬프트 평가

**부분 작성 · 개정 프롬프트의 답변 함수**

**제공:** 기존 프롬프트에서 "구절을 바꿔 쓰거나 요약하지 마세요." 한 문장만 바꾼 `HR_SYSTEM_V2`와 `hr_chain_v2`입니다.

**작성:**

1. `hr_answer_v2(question)`: `hr_answer`와 같은 순서(검색, 체인 호출, 부분 `text`를 이은 `answer`와 `searched_ids` 넣기)로 쓰되 `hr_chain` 대신 `hr_chain_v2`를 호출합니다.
2. `hr_v2_params`: `dict(hr_params)`로 복사한 뒤 `prompt_version`만 `"v2"`로 바꿉니다. 복사하지 않고 `hr_v2_params = hr_params`로 쓰면 기준선에 담긴 `hr_params`까지 함께 바뀝니다.
3. `evaluate_run`으로 `hr-v2` run을 평가해 결과를 `hr_v2_result`에 담고, `make_report`로 만든 보고서를 `hr_v2_report`에 담습니다.

In [ ]:
# 기존 프롬프트의 "구절을 바꿔 쓰거나 요약하지 마세요."를 "인용 구절은 읽기 쉽게 다듬어 쓰세요."로 바꿨습니다.
HR_SYSTEM_V2 = ("당신은 회사 인사팀 상담 챗봇입니다. 고용노동부 유연근무 활용 매뉴얼 발췌로만 답하세요. "
                "답변을 부분(segments)으로 나눠 쓰고, 매뉴얼에서 가져온 사실을 쓴 부분마다 근거 ID와 그 절에서 글자 그대로 복사한 구절을 인용하세요. "
                "인용 구절은 읽기 쉽게 다듬어 쓰세요. 적용 대상, 요건, 기간, 예외를 빠뜨리지 마세요. "
                "매뉴얼에 답이 없으면 '제공된 매뉴얼에서 확인할 수 없습니다.'라는 부분 하나만 인용 없이 쓰세요.")
hr_prompt_v2 = ChatPromptTemplate.from_messages([("system", HR_SYSTEM_V2), ("human", "질문: {question}\n\n매뉴얼 발췌:\n{context}")])

hr_chain_v2 = hr_prompt_v2 | llm.with_structured_output(CitedAnswer)


@mlflow.trace
def hr_answer_v2(question):
    """개정 프롬프트(hr_chain_v2)로 답한다는 것 말고는 hr_answer 와 같습니다."""
    # 검색 -> hr_chain_v2 호출 -> dict 로 바꾸고 answer(부분 text 잇기)와 searched_ids 넣기를 작성하세요.
    ...


hr_v2_params = ...

hr_v2_result = ...
hr_v2_report = ...

print("개정 프롬프트 지표:", hr_v2_report["metrics"])

### 7-5. 대조 run과 개정 프롬프트 판정

본 실습의 테스트 파일을 `hr_eval_ci/tests/`로 복사합니다. 채점 함수 이름과 사례는 기준선 파일·결과 파일에서 읽으므로 테스트 파일은 고치지 않습니다.

판정 대상은 개정 프롬프트(`hr-v2`)이고, 비교하려고 같은 설정으로 평가한 둘째 run(`hr-v1-b`)의 결과를 먼저 판정합니다. 두 보고서를 차례로 저장하고 pytest를 실행합니다.

In [ ]:
import shutil

(hr_dir / "tests").mkdir(parents=True, exist_ok=True)
shutil.copy(project_dir / "tests" / "test_rag_regression.py", hr_dir / "tests" / "test_rag_regression.py")

hr_results_path = hr_dir / "reports" / "results.json"
hr_results_path.parent.mkdir(parents=True, exist_ok=True)

# 같은 설정으로 평가한 둘째 run(hr-v1-b)의 결과입니다.
hr_results_path.write_text(json.dumps(hr_repeat_reports[1], ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("[같은 설정 hr-v1-b]")
run_pytest(hr_dir)

# 개정 프롬프트의 결과입니다.
hr_results_path.write_text(json.dumps(hr_v2_report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
print("\n[개정 프롬프트]")
run_pytest(hr_dir)

In [ ]:
# 개정 프롬프트에서 인용 검증에 실패한 사례와 실패 이유를 출력합니다.
for _, row in hr_v2_result.result_df.sort_values("case_id/value").iterrows():
    if row["citation_valid/value"] == False:
        print(row["case_id/value"], "|", row["citation_valid/rationale"])

`hr-v1-b`는 기준선을 만들 때 쓴 run이라 허용 하락폭과 사례별 회귀 테스트는 통과합니다.

`[같은 설정 hr-v1-b]`와 `[개정 프롬프트]`의 `종료 코드:` 줄을 비교합니다. 개정 프롬프트에서 `correctness`는 기준선과 비슷한데 `citation_valid`가 떨어졌고 바로 위 셀의 실패 이유가 `원문에 없는 문구`라면, 답의 내용은 맞지만 사용자가 원문에서 인용 구절을 찾을 수 없게 된 것입니다.

실패한 테스트 이름과 이유를 읽고, 이 변경을 받아들일지와 그 이유를 한 문장으로 정리해 봅니다.

## 8. 핵심 정리

- **run 기록:** `start_run(run_name=...)` 안에서 `log_param`으로 조건을 남기고 평가하면 실행 조건과 평가 결과가 한 run에 남습니다.
- **흔들림:** 같은 설정을 반복 평가해 사례마다 통과 횟수를 세면 판정이 흔들리는 사례와 늘 통과한 사례가 갈립니다.
- **기준선:** 사람이 승인한 run입니다. MLflow에는 champion 태그로, CI에는 `baseline.json`으로 남기며, 갱신은 검토와 승인을 거칩니다.
- **판정 규칙:** 절대 하한, 허용 하락폭(지표 흔들림 + 사례 1건만큼의 여유), 사례별 회귀, 평가 완전성을 함께 봅니다.
- **변경 비교:** `search_runs`로 run을 표로 모으고, `case_id`로 기준선 run과 사례별 판정을 맞춰 늘 통과한 사례가 실패했는지 봅니다.
- **pytest 회귀 테스트:** 평가 결과 파일을 `test_` 함수의 `assert`로 판정하고, 실패하면 종료 코드가 0이 아니게 됩니다. DeepEval `assert_test`는 테스트 실행 중에 판사를 불러 사례 하나에 통과 점수를 적용합니다.
- **CI:** 변경 요청마다 smoke, 매일 예약 실행에서 full을 돌리고, API 키는 비밀 변수로 넘기며, 결과는 요약과 산출물로 남깁니다.

다음 단원에서는 답변이 나가기 전에 위험한 입력과 출력을 막는 가드레일을 다룹니다.